In [ ]:
!pip install -Uqq fastai

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pandas # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Import required FastAI libraries
from fastai import *
from fastai.vision.all import *
from fastcore.all import *

# For image manipulation
from PIL import Image

In [ ]:
# Get kaggle data to path
path = Path('/kaggle/input/digit-recognizer')
path.ls()

In [ ]:
# Visualize data using pandas
train_df = pandas.read_csv(path/'train.csv')
train_df

# General functions

In [ ]:
def create_path(path):
    path = Path(path)
    if not os.path.exists(path):
        os.makedirs(path)
    
    return path

In [ ]:
def convert_csv_data_to_images(df, path): 
    # Iterate through each row
    for index, row in df.iterrows():
        # Check if the row has a 'label' column
        if 'label' in row.index:
            label = row['label']
            pixels = row.drop('label').values.astype(np.uint8)
            image_path = path / f'{label}_{index}.jpg'
        else:
            pixels = row.values.astype(np.uint8)
            image_path = path / f'{index}.jpg'
        
        # Reshape into 28x28 array
        pixels = pixels.reshape((28, 28))
        
        # Create an image from the array
        image = Image.fromarray(pixels, 'L')  # 'L' indicates grayscale
        image.save(image_path)
    
    number_files = len(os.listdir(path))
    print(number_files)

# Convert a CSV data to JPG images

In [ ]:
path = create_path('/kaggle/train')

In [ ]:
convert_csv_data_to_images(train_df, path)

In [ ]:
Image.open('/kaggle/train/5_25624.jpg')

# Build model

In [ ]:
mnist_data_block = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=using_attr(RegexLabeller(r'(.+)_\d+.*$'), 'name')
)

In [ ]:
dls = mnist_data_block.dataloaders(path)
dls.show_batch(max_n=32)

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy)

In [ ]:
learn.lr_find()

In [ ]:
learn.fine_tune(3)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)

interp.plot_confusion_matrix()

In [ ]:
interp.plot_top_losses(6, nrows=2)

# Test against test set

In [ ]:
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
test_df

In [ ]:
path = create_path('/kaggle/test')

In [ ]:
convert_csv_data_to_images(test_df, path)

In [ ]:
paths = [str(f) for f in path.iterdir()]
paths.sort(key=lambda f: int(Path(f).stem))
paths[0]

In [ ]:
test_dl = learn.dls.test_dl(paths)

In [ ]:
predictions = learn.get_preds(dl=test_dl)

In [ ]:
predictions[0][0]

In [ ]:
sample_df = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')
sample_df

In [ ]:
sample_df['Label'] = np.argmax(predictions[0], axis=1)
sample_df

In [ ]:
image = Image.open('/kaggle/test/1.jpg')
image

In [ ]:
prediction,idx, probabilities = learn.predict(image)

prediction

In [ ]:
sample_df.to_csv('result.csv', index=False)